In [2]:
# 정상 이미지 라벨 생성
def generate_normal_labels_from_csv(csv_path, output_json_path):
    import pandas as pd, json
    df = pd.read_csv(csv_path)
    labels = [{"frame_idx": i, "is_correct": 1, "wrong_joints": []} for i in range(len(df))]
    with open(output_json_path, "w") as f:
        json.dump(labels, f, indent=2)

In [ ]:
# 오류 데이터 생성: CSV에 오류 삽입기 + 라벨 생성기 민선이꺼
def corrupt_with_alternating_pattern(input_csv_path, output_csv_path, output_label_json,
                                      start_frame=0, end_frame=1000,
                                      magnitude=70, window_size=40): #swimming은 평균 4초 -> 120 프레임 40 정도로 해서 오류-정상-오류로 만듦
                                                                    
    import pandas as pd, numpy as np, json, random

    JOINT_GROUPS = {
        "RShoulder": ["RShoulder", "RElbow", "RWrist"],
        "LShoulder": ["LShoulder", "LElbow", "LWrist"],
        "RHip": ["RHip", "RKnee", "RAnkle"],
        "LHip": ["LHip", "LKnee", "LAnkle"]
        #"Neck": ["Neck", "Head"]
    }

    df = pd.read_csv(input_csv_path)
    num_frames = len(df)

    labels = [{"frame_idx": i, "is_correct": 1, "wrong_joints": []} for i in range(num_frames)]

    toggle = True 

    for cur in range(start_frame, end_frame, window_size):
        end = min(cur + window_size, end_frame, num_frames)

        if toggle:
            #오류 삽입
            error_joint = random.choice(list(JOINT_GROUPS.keys()))
            error_joints = JOINT_GROUPS[error_joint]

            for i in range(cur, end):
                labels[i]["is_correct"] = 0
                labels[i]["wrong_joints"] = error_joints
                for joint in error_joints:
                    df.loc[i, f"{joint}_x"] += np.random.uniform(-magnitude, magnitude)
                    df.loc[i, f"{joint}_y"] += np.random.uniform(-magnitude, magnitude)
        

        toggle = not toggle  #다음 구간은 반대로

    df.to_csv(output_csv_path, index=False)
    with open(output_label_json, "w") as f:
        json.dump(labels, f, indent=2)

In [ ]:
#MediaPipe 기반 스켈레톤 이미지 생성기
def generate_skeleton_images_from_pose_csv(csv_path, output_dir):
    import pandas as pd, cv2, numpy as np, os

    keypoints = ['Head', 'LShoulder', 'RShoulder', 'LElbow', 'RElbow',
                 'LWrist', 'RWrist', 'LHip', 'RHip', 'LKnee', 'RKnee',
                 'LAnkle', 'RAnkle', 'Neck', 'Hip']

    skeleton_lines = [
        ('Head', 'Neck'),
        ('Neck', 'LShoulder'), ('Neck', 'RShoulder'),
        ('LShoulder', 'LElbow'), ('LElbow', 'LWrist'),
        ('RShoulder', 'RElbow'), ('RElbow', 'RWrist'),
        ('LShoulder', 'LHip'), ('RShoulder', 'RHip'),
        ('LHip', 'LKnee'), ('LKnee', 'LAnkle'),
        ('RHip', 'RKnee'), ('RKnee', 'RAnkle'),
        ('LHip', 'RHip'), ('Hip', 'Neck')
    ]

    def center_skeleton(image):
        gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
        points = cv2.findNonZero(255 - gray)
        if points is None:
            return image
        x, y, w, h = cv2.boundingRect(points)
        cx, cy = x + w // 2, y + h // 2
        h_img, w_img = image.shape[:2]
        dx, dy = (w_img // 2 - cx), (h_img // 2 - cy)
        M = np.float32([[1, 0, dx], [0, 1, dy]])
        return cv2.warpAffine(image, M, (w_img, h_img), borderValue=(255, 255, 255))

    df = pd.read_csv(csv_path)
    os.makedirs(output_dir, exist_ok=True)

    for idx, row in df.iterrows():
        canvas = np.ones((2000, 2000, 3), dtype=np.uint8) * 255
        coords = {k: (int(row[f'{k}_x']), int(row[f'{k}_y'])) for k in keypoints}

        #선 그리기
        for pt1, pt2 in skeleton_lines:
            cv2.line(canvas, coords[pt1], coords[pt2], (0, 0, 0), 3)

        #점 ㅉ찍기
        for x, y in coords.values():
            cv2.circle(canvas, (x, y), 6, (0, 0, 255), -1)

        #이미지 중앙정렬
        centered = center_skeleton(canvas)

        img_name = row['img_name'].split('.')[0] if 'img_name' in row else f"frame_{idx:03d}"
        save_path = os.path.join(output_dir, f"{img_name}_skeleton.png")
        cv2.imwrite(save_path, centered)

    print(f"생성 완료: {output_dir}")


In [ ]:
import os
import pandas as pd
import json

def generate_normal_labels_from_csv(csv_path, output_json_path):
    df = pd.read_csv(csv_path)
    labels = [{"frame_idx": i, "is_correct": 1, "wrong_joints": []} for i in range(len(df))]
    with open(output_json_path, "w") as f:
        json.dump(labels, f, indent=2)


base_dir = r"C:\Soop\aiApplication\aAi\skeleton_images\Hundred"

#pose_keypoints.csv 경로 수집, 라벨 생성
for root, dirs, files in os.walk(base_dir):
    for file in files:
        if file == "pose_keypoints.csv":
            csv_path = os.path.join(root, file)
            output_json_path = os.path.join(root, "labels.json")
            generate_normal_labels_from_csv(csv_path, output_json_path)
            print(f"정상 라벨 생성 완료: {output_json_path}")


정상 라벨 생성 완료: C:\Soop\aiApplication\aAi\skeleton_images\Hundred\actorP061(1)\labels.json
정상 라벨 생성 완료: C:\Soop\aiApplication\aAi\skeleton_images\Hundred\actorP061(1)_4\labels.json
정상 라벨 생성 완료: C:\Soop\aiApplication\aAi\skeleton_images\Hundred\actorP061(2)\labels.json
정상 라벨 생성 완료: C:\Soop\aiApplication\aAi\skeleton_images\Hundred\actorP061(2)_4\labels.json
정상 라벨 생성 완료: C:\Soop\aiApplication\aAi\skeleton_images\Hundred\actorP061(3)\labels.json
정상 라벨 생성 완료: C:\Soop\aiApplication\aAi\skeleton_images\Hundred\actorP061(4)\labels.json
정상 라벨 생성 완료: C:\Soop\aiApplication\aAi\skeleton_images\Hundred\actorP061(5)\labels.json
정상 라벨 생성 완료: C:\Soop\aiApplication\aAi\skeleton_images\Hundred\actorP061(6)\labels.json
정상 라벨 생성 완료: C:\Soop\aiApplication\aAi\skeleton_images\Hundred\actorP061(7)\labels.json
정상 라벨 생성 완료: C:\Soop\aiApplication\aAi\skeleton_images\Hundred\actorP061(8)\labels.json
정상 라벨 생성 완료: C:\Soop\aiApplication\aAi\skeleton_images\Hundred\actorP062(1)\labels.json
정상 라벨 생성 완료: C:\Soop\aiAppli

KeyboardInterrupt: 

In [ ]:
import os

#메인 로직
base_dir = r"C:\Soop\aiApplication\aAi\skeleton_images\Teaser"
wrong_suffix = "_wrong"

#전체 pose_keypoints.csv 탐색, 오류 데이터 생성
for root, dirs, files in os.walk(base_dir):
    for file in files:
        if file == "pose_keypoints.csv":
            csv_path = os.path.join(root, file)
            folder = os.path.dirname(csv_path)
            base_name = os.path.basename(folder)
            parent_dir = os.path.dirname(folder)

            wrong_dir = os.path.join(parent_dir, base_name + wrong_suffix)
            os.makedirs(wrong_dir, exist_ok=True)
            corrupted_csv_path = os.path.join(wrong_dir, "pose_keypoints.csv")
            label_json_path = os.path.join(wrong_dir, "labels.json")
            image_output_dir = wrong_dir

            corrupt_with_alternating_pattern(
                input_csv_path=csv_path,
                output_csv_path=corrupted_csv_path,
                output_label_json=label_json_path,
                magnitude=70,
                window_size=40
            )
            generate_skeleton_images_from_pose_csv(corrupted_csv_path, image_output_dir)

            print(f"오류 데이터 생성 완료: {wrong_dir}")


생성 완료: C:\Soop\aiApplication\aAi\skeleton_images\Teaser\actorP061(1)_4_wrong
오류 데이터 생성 완료: C:\Soop\aiApplication\aAi\skeleton_images\Teaser\actorP061(1)_4_wrong
생성 완료: C:\Soop\aiApplication\aAi\skeleton_images\Teaser\actorP061(10)_4_wrong
오류 데이터 생성 완료: C:\Soop\aiApplication\aAi\skeleton_images\Teaser\actorP061(10)_4_wrong
생성 완료: C:\Soop\aiApplication\aAi\skeleton_images\Teaser\actorP061(11)_4_wrong
오류 데이터 생성 완료: C:\Soop\aiApplication\aAi\skeleton_images\Teaser\actorP061(11)_4_wrong
생성 완료: C:\Soop\aiApplication\aAi\skeleton_images\Teaser\actorP061(2)_4_wrong
오류 데이터 생성 완료: C:\Soop\aiApplication\aAi\skeleton_images\Teaser\actorP061(2)_4_wrong
생성 완료: C:\Soop\aiApplication\aAi\skeleton_images\Teaser\actorP061(3)_4_wrong
오류 데이터 생성 완료: C:\Soop\aiApplication\aAi\skeleton_images\Teaser\actorP061(3)_4_wrong
생성 완료: C:\Soop\aiApplication\aAi\skeleton_images\Teaser\actorP061(4)_4_wrong
오류 데이터 생성 완료: C:\Soop\aiApplication\aAi\skeleton_images\Teaser\actorP061(4)_4_wrong
생성 완료: C:\Soop\aiApplication\a